# 01 — Quickstart (library mode)

End-to-end tour of `sonar-probe` as a Python library:

1. synthesize a tiny audio corpus + plain-text manifest
2. extract 1024-d SONAR embeddings into a Kaldi ark/scp store
3. load the store back and compare two segments

**Offline mode.** Each notebook injects a deterministic `FakeSpeechEncoder` so the examples run without downloading SONAR weights. To use real SONAR, delete the `pool=` / `text_encoder=` arguments where noted.

In [ ]:
import tempfile, wave, hashlib
from pathlib import Path

import numpy as np

from sonar_probe import ExtractConfig, CompareConfig, extract, compare, load_store
from sonar_probe.sources import get_source
from sonar_probe.sinks import get_sink
from sonar_probe.engine.speech import SpeechEncoderPool
from sonar_probe.engine.text import TextEncoder

WORK = Path(tempfile.mkdtemp(prefix="sonar-probe-quickstart-"))
print("workdir:", WORK)

## Deterministic fake encoders

The real SONAR encoders return `(B, 1024) float32`. Any object with the same `encode(...)` signature works — this is the `encoder_factory` injection point used throughout the test suite.

The fakes below hash their input so identical inputs → identical embeddings; different inputs → different embeddings. That is enough to see cosine similarity move.

In [ ]:
DIM = 1024

def _hash_vec(key: bytes) -> np.ndarray:
    rng = np.random.default_rng(int.from_bytes(hashlib.sha256(key).digest()[:8], "big"))
    v = rng.standard_normal(DIM).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-9)

class FakeSpeechEncoder:
    def encode(self, waveforms):
        return np.stack([_hash_vec(w.tobytes()) for w in waveforms]).astype(np.float32)

class FakeTextBackend:
    def encode(self, texts, lang):
        return np.stack([_hash_vec(f"{lang}|{t}".encode()) for t in texts]).astype(np.float32)

fake_pool = SpeechEncoderPool(device="cpu", encoder_factory=lambda lang, device: FakeSpeechEncoder())
fake_text = TextEncoder(device="cpu", encoder_factory=lambda device: FakeTextBackend())

## Synthesize a tiny corpus

Two sine-tone WAVs at 16 kHz, with a 5-row segments manifest.

In [ ]:
SR = 16000

def write_tone(path: Path, hz: float, seconds: float) -> None:
    t = np.arange(int(SR * seconds)) / SR
    x = (0.3 * np.sin(2 * np.pi * hz * t) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(x.tobytes())

audio_dir = WORK / "audio"; audio_dir.mkdir()
write_tone(audio_dir / "rec01.wav", 440.0, 6.0)
write_tone(audio_dir / "rec02.wav", 880.0, 6.0)

(WORK / "wav.scp").write_text(f"rec01 {audio_dir/'rec01.wav'}\nrec02 {audio_dir/'rec02.wav'}\n")
(WORK / "segments.tsv").write_text(
    "file_id\tchannel\tspkid\tstart\tend\tlang\n"
    "rec01\t0\tspk1\t0.0\t2.0\teng\n"
    "rec01\t0\tspk1\t2.0\t4.0\teng\n"
    "rec01\t0\tspk1\t4.0\t6.0\teng\n"
    "rec02\t0\tspk2\t0.0\t3.0\teng\n"
    "rec02\t0\tspk2\t3.0\t6.0\teng\n"
)
print("wrote", sorted(p.name for p in WORK.iterdir()))

## Extract into a Kaldi ark/scp store

`extract()` iterates a `SegmentSource`, decodes each segment, calls the speech encoder, and writes `(id, embedding)` pairs to the sink. Soft errors land in `failures.jsonl` beside the output.

In [ ]:
src = get_source("plain_text", input=WORK / "segments.tsv", wav_scp=WORK / "wav.scp")
snk = get_sink("kaldi_ark", output=WORK / "store")
cfg = ExtractConfig(device="cpu", batch_size=4, speech_lang="eng")

# For real SONAR, drop the `pool=` kwarg below.
res = extract(src, snk, cfg, pool=fake_pool)
print(f"written={res.written} skipped={res.skipped} failures={res.failures_path}")
print("store files:", sorted(p.name for p in (WORK / 'store').iterdir()))

## Load the store

`load_store(path)` sniffs the layout and returns a store object exposing `__iter__`, `__getitem__`, `__len__`, `ids()`, and `matrix(ids=None) -> (list[str], (N, 1024) float32)`.

In [ ]:
store = load_store(WORK / "store")
ids, mat = store.matrix()
print("count:", len(store), "matrix:", mat.shape, "dtype:", mat.dtype)
print("ids:", ids)
print("first record modality/lang:", store[ids[0]].modality, store[ids[0]].lang)

## Compare two segments

`compare()` resolves each operand to a `(N, 1024)` matrix, L2-normalizes, and returns the full `(len(a), len(b))` cosine matrix. For a 1×1 result just pull `scores[0, 0]`.

In [ ]:
a_id, b_id = ids[0], ids[1]      # two segments from rec01
c_id = ids[-1]                    # a segment from rec02

cfg = CompareConfig(device="cpu")
same_speaker = compare(f"store:{WORK/'store'}#{a_id}", f"store:{WORK/'store'}#{b_id}",
                       cfg, pool=fake_pool, text_encoder=fake_text)
cross = compare(f"store:{WORK/'store'}#{a_id}", f"store:{WORK/'store'}#{c_id}",
                cfg, pool=fake_pool, text_encoder=fake_text)

print(f"within rec01 cosine = {float(same_speaker.scores[0,0]):+.4f}")
print(f"rec01 ↔ rec02 cosine = {float(cross.scores[0,0]):+.4f}")

## Next steps

- `02_cli_tour.ipynb` — same workflow driven from the `sonar-probe` CLI.
- `03_input_sources.ipynb` — the three built-in sources (plain_text, kaldi, hf) side by side.
- `04_compare_cross_modal.ipynb` — text-vs-speech compare, inline segments, `top_k` / `threshold` filtering, matrix output.